# HR DATA CLEANING
 **Project Objectives**
- This project transforms raw, unstructured HR data into an analysis-ready dataset by leveraging modular and reusable Python functions.

**Phase 1. Imported the Required Libraries and Loaded Dataset**

In [1281]:
#Import required libraries
import pandas as pd
import numpy as np

#Load Dataset
hrdata= pd.read_excel(r"C:\Users\HP\OneDrive\Desktop\LuxDev Tutorials\Python Projects\HR Data Analysis Project.ipynb\HR_Dirty_Data.xlsx")

**Phase 2. Initial Data Quality Assessment Report**

In [1282]:
quality_report = pd.DataFrame({
    "Column": hrdata.columns,
    "Total Columns": len(hrdata.columns),
    "Data_Type": hrdata.dtypes.astype(str),
    "Total_Rows": len(hrdata),
    "Missing_Values": hrdata.isna().sum().values,
    "Missing_Percentage": (hrdata.isna().mean() * 100).round(2).values,
    "Unique_Values": hrdata.nunique().values,
})

quality_report


,Column,Total Columns,Data_Type,Total_Rows,Missing_Values,Missing_Percentage,Unique_Values
Employee ID,Employee ID,21,object,876,7,0.80,852
First Name,First Name,21,str,876,11,1.26,123
Last Name,Last Name,21,str,876,9,1.03,84
Department,Department,21,str,876,12,1.37,22
Salary,Salary,21,object,876,9,1.03,855
Hire Date,Hire Date,21,object,876,12,1.37,769
Age,Age,21,object,876,7,0.80,59
Gender,Gender,21,str,876,20,2.28,9
Performance Score,Performance Score,21,object,876,15,1.71,19
Full-Time,Full-Time,21,str,876,9,1.03,10


**Phase 3. Data Cleaning Steps Employed**

**Step 1: Rename  and Standardize Column Heads**
- used the .rename() method:

In [1283]:
print(f"Current Column Names Heading\n",hrdata.columns)

Current Column Names Heading
 Index(['Employee ID', 'First Name', 'Last Name', 'Department', 'Salary',
       'Hire Date', 'Age', 'Gender', 'Performance Score', 'Full-Time', 'Bonus',
       'Marital Status', 'Education Level', 'Work Experience (Years)',
       'Employee Type', 'Office Location', 'Project Count',
       'Last Promotion Year', 'Remote Work Status', 'Annual Training Hours',
       'Manager Feedback Score'],
      dtype='str')


Column by column  renaming

In [1284]:
hrdata = hrdata.rename(columns={'Employee ID': 'employee_id'})
hrdata = hrdata.rename(columns={'First Name': 'first_name'})
hrdata = hrdata.rename(columns={'Last Name': 'last_name'})

A function to rename heads

In [1285]:
def clean_column_names(hrdata):
    """Converts all column headers to lowercase, replaces spaces with underscores, 
    and removes special characters automatically."""
    hrdata.columns = (
        hrdata.columns
        .str.lower()
        .str.strip()
        .str.replace('[^a-z0-9_]', '_', regex=True)  # Replaces spaces, hyphens, & () with _
        .str.replace('_+', '_', regex=True)          # Fixes double underscores like __
        .str.strip('_')                              # Removes trailing/leading underscores
    )
    return hrdata

# call the function
hrdata = clean_column_names(hrdata)

#Final Columns
print(f"Cleaned Headings\n",hrdata.columns)

Cleaned Headings
 Index(['employee_id', 'first_name', 'last_name', 'department', 'salary',
       'hire_date', 'age', 'gender', 'performance_score', 'full_time', 'bonus',
       'marital_status', 'education_level', 'work_experience_years',
       'employee_type', 'office_location', 'project_count',
       'last_promotion_year', 'remote_work_status', 'annual_training_hours',
       'manager_feedback_score'],
      dtype='str')


**Step 2: Identify and Remove Duplicates**
- There are 7 duplicated rows in the dataset

In [1286]:
# Check duplicate rows
print("Duplicates before:", hrdata.duplicated().sum())

# Remove duplicate rows
hrdata = hrdata.drop_duplicates()

#OR hrdata.drop_duplicates(inplace=True)

# Verify
print("Duplicates after:", hrdata.duplicated().sum())

Duplicates before: 7
Duplicates after: 0


**Remove Missing Values in employee_id**

In [1287]:
# 1. Inspect rows with missing employee_id (subsets the DataFrame for viewing)
missing_rows = hrdata[hrdata['employee_id'].isnull()]

# 2. Permanently drop rows where employee_id is null (ASSIGN BACK to update hrdata)
hrdata = hrdata.dropna(subset=["employee_id"])

# 3. Verify that nulls are now 0
print(hrdata["employee_id"].isnull().sum())  # Now correctly returns 0

0


**Step 3. Categorical Columns Cleaning and Standardization**

- Fix Category Misspellings, Typos and missing values

In [1288]:
# Select columns containing text/categorical data
# These columns are often where inconsistent spelling or spacing occurs
categorical_columns = hrdata.select_dtypes(include=['object', 'string', 'category']).columns

# Display the categorical column names
categorical_columns

Index(['employee_id', 'first_name', 'last_name', 'department', 'salary',
       'hire_date', 'age', 'gender', 'performance_score', 'full_time', 'bonus',
       'marital_status', 'education_level', 'work_experience_years',
       'employee_type', 'office_location', 'project_count',
       'last_promotion_year', 'remote_work_status', 'manager_feedback_score'],
      dtype='str')

"first_name","last_name"

In [1289]:
hrdata['first_name']=hrdata['first_name'].str.strip().str.title()

#Check data type
print(type('first_name'))

hrdata['last_name']=hrdata['last_name'].str.strip().str.title()

#Check data type
print(type('last_name'))


<class 'str'>
<class 'str'>


'department'

In [1290]:
# Convert to tiltle case and remove leading and training spaces
hrdata['department']=hrdata['department'].str.strip().str.title()


In [1291]:
# Standardize attributes
# List unique categories in 'Gender'
print(hrdata["department"].unique())

# See each category and how many times it appears
print(hrdata["department"].value_counts())



<ArrowStringArray>
[              'Hr',              'H.R',               'It',
            'Sales',       'Operations',        'Info Tech',
          'Finance',        'Marketing',             'Sale',
         'Finanace',  'Human Resources',  'Humna Resources',
   'Human Resource',      'Humman Res.',         'Markting',
 'Information Tech',                nan,        'Operatons',
              'Ops',              'I.T']
Length: 20, dtype: str
department
Finance             150
Hr                  131
It                  129
Operations          120
Sales               119
Marketing           108
H.R                  12
Info Tech            12
Human Resources      12
Human Resource        9
Operatons             8
Ops                   7
Markting              6
Information Tech      6
Sale                  5
Finanace              5
Humman Res.           4
I.T                   4
Humna Resources       3
Name: count, dtype: int64


In [1292]:
#  Standardize categories only to  Finance,  Human Resource ,Operations,Marketing  ,Information Tech, Sales,and Unknown
# Replace 'Non-Binary', 'Other', and 'None' to 'Unknown'
hrdata["department"] = hrdata["department"].replace({
    "Finanace": "Finance", 
    "Hr": "Human Resources", "H.R": "Human Resources","Humna Resources": "Human Resources","Human Resource": "Human Resources","Humman Res.": "Human Resources",
    "Markting": "Marketing",
    "Ops": "Operatons","Operatons": "Operations",
    "Sale": "Sales",
    "I.T": "Information Tech","Info Tech": "Information Tech","I.T": "Information Tech","It": "Information Tech",
    "nan":"Unknown"})

# Verify the result
print(hrdata["department"].value_counts())

#Check data type
print(type('department'))

department
Human Resources     171
Finance             155
Information Tech    151
Operations          128
Sales               124
Marketing           114
Operatons             7
Name: count, dtype: int64
<class 'str'>


In [1293]:
# Check current null count
print("Nulls before:", hrdata['department'].isnull().sum())

# Permanently drop rows where department is null
hrdata['department'] = hrdata['department'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['department'].isnull().sum())

Nulls before: 12
Nulls after: 0


'gender'

In [1294]:
# Convert to tiltle case and remove leading and training spaces
hrdata['gender']=hrdata['gender'].str.strip().str.title()

# List unique categories in 'Gender'
print(hrdata["gender"].unique())

# See each category and how many times it appears
print(hrdata["gender"].value_counts())

<ArrowStringArray>
['Female', 'Male', 'Femle', 'M', nan, 'F', 'Prefer Not Say']
Length: 7, dtype: str
gender
Male              407
Female            403
Femle              11
Prefer Not Say      9
F                   8
M                   4
Name: count, dtype: int64


In [1295]:
#  Standardize categories only to  Male,  Female ,Unknown ,Information Tech, Sales,and Unknown
# Replace 'nan', 'Prefer Not Say' to 'Unknown'
hrdata["gender"] = hrdata["gender"].replace({
    "M": "Male", 
    "F": "Female", "Femle": "Female",
    "Prefer Not Say": "Unknown","nan": "Unknown"
    })

# Verify the result
print(hrdata["gender"].value_counts())

#Check data type
print(type('gender'))

gender
Female     422
Male       411
Unknown      9
Name: count, dtype: int64
<class 'str'>


In [1296]:
# Check current null count
print("Nulls before:", hrdata['gender'].isnull().sum())

# Permanently drop rows where gender is null
hrdata['gender'] = hrdata['gender'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['gender'].isnull().sum())

Nulls before: 20
Nulls after: 0


full_time

In [1297]:
# Convert to tiltle case and remove leading and training spaces
hrdata['full_time']=hrdata['full_time'].str.strip().str.title()

# Replace Missing Values
hrdata['full_time'] = hrdata['full_time'].fillna('Unknown')

# See each category and how many times it appears
print(hrdata["full_time"].value_counts())

full_time
Yes          429
No           402
N             10
Unknown        9
Y              5
Part-Time      4
Full Time      3
Name: count, dtype: int64


In [1298]:
# Standardize attributes
hrdata["full_time"]=hrdata["full_time"].replace({
    "Y":"Yes","Full Time":"Yes",
    "N":"No","Part-Time":"No",
    "nan":"Unknown"
})

# See each category and how many times it appears
print(hrdata["full_time"].value_counts())

#Check data type
print(type('first_time'))

full_time
Yes        437
No         416
Unknown      9
Name: count, dtype: int64
<class 'str'>


In [1299]:
# Check current null count
print("Nulls before:", hrdata['full_time'].isnull().sum())

# Address missing values
hrdata['full_time'] = hrdata['full_time'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['full_time'].isnull().sum())

Nulls before: 0
Nulls after: 0


'marital_status'

In [1300]:
#Remove  leading and trailing  spaces and change to title case
#hrdata['marital_status'].str.title().str.strip() # transform only
hrdata['marital_status'] = hrdata['marital_status'].str.strip().str.title() # transform and save

#Identify  unique categories in the column
print(hrdata['marital_status'].unique())

#Count unique categories in the column
hrdata['marital_status'].value_counts()

<ArrowStringArray>
['Widowed', 'Married', 'Single', 'Divorced', nan, 'Widwowed', 'Maried']
Length: 7, dtype: str


marital_status
Married     219
Widowed     214
Single      209
Divorced    199
Widwowed      7
Maried        4
Name: count, dtype: int64

In [1301]:
#Standardize inconsistent categories
hrdata['marital_status']=hrdata['marital_status'].replace({
    "Widwowed":"Widowed",
    "maried":"Married",
    "single":"Single",
    "nan":"Unknown"
})

#Check data type
print(type('marital_status'))

<class 'str'>


In [1302]:
# Check current null count
print("Nulls before:", hrdata['marital_status'].isnull().sum())

# Address Missing Values
hrdata['marital_status'] = hrdata['marital_status'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['marital_status'].isnull().sum())

Nulls before: 10
Nulls after: 0


In [1303]:
hrdata.columns

Index(['employee_id', 'first_name', 'last_name', 'department', 'salary',
       'hire_date', 'age', 'gender', 'performance_score', 'full_time', 'bonus',
       'marital_status', 'education_level', 'work_experience_years',
       'employee_type', 'office_location', 'project_count',
       'last_promotion_year', 'remote_work_status', 'annual_training_hours',
       'manager_feedback_score'],
      dtype='str')

education_level

In [1304]:
#Remove  leading and trailing  spaces and change to title case
hrdata['education_level']=hrdata['education_level'].str.title().str.strip()

#Identify  unique categories in the column
print(hrdata['education_level'].unique())

#Count unique categories in the column
hrdata['education_level'].value_counts()

<ArrowStringArray>
[        'Phd', 'High School', 'Associate'S',  'Bachelor'S',    'Bachelor',
    'High Sch',    'Master'S',     'Masters',           nan,   'Bachelors',
  'Associates',         'Msc']
Length: 12, dtype: str


education_level
Associate'S    178
Bachelor'S     170
Master'S       165
High School    161
Phd            152
High Sch         6
Bachelors        6
Associates       6
Bachelor         5
Masters          5
Msc              2
Name: count, dtype: int64

In [1305]:
#Standardize inconsistent categories
hrdata['education_level']=hrdata['education_level'].replace({
    "PHD":"PhD","phd":"PhD",
    "Associates":"Associate's",
    "high school":"High School","High Sch":"High School",
    "Bachelor":"Bachelors","Bachelor's":"Bachelors",
    "Master's":"Masters", "MSc":"Masters",
    "nan":"Unknown"
})

#Check data type
print(type('education_level'))

<class 'str'>


In [1306]:
# Check current null count
print("Nulls before:", hrdata['education_level'].isnull().sum())

# Address missing values
hrdata['education_level'] = hrdata['education_level'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['education_level'].isnull().sum())

Nulls before: 6
Nulls after: 0


'employee_type'

In [1307]:
# Convert the column to title case and remove leading and trailing spaces
hrdata['employee_type']=hrdata['employee_type'].str.title().str.strip()

# List usnique categories in the column
print(hrdata['employee_type'].unique())

#Count all unique categories
hrdata['employee_type'].value_counts()


<ArrowStringArray>
[    'Intern',   'Contract',  'Permanent',  'Temporary',      'Inten',
       'Perm', 'Contractor',    'Contrct',          nan]
Length: 9, dtype: str


employee_type
Intern        280
Permanent     273
Contract      270
Temporary       9
Perm            8
Inten           5
Contractor      5
Contrct         5
Name: count, dtype: int64

In [1308]:
#Standardize insconsistent categories

hrdata['employee_type']=hrdata['employee_type'].replace({
    "Inten":"Intern",
    "Perm":"Permanent",
    "Contractor":"Contract","Contrct":"Contract",
    "nana":"Unknown"
})

#Check data type
print(type('employee_type'))

<class 'str'>


In [1309]:
# Check current null count
print("Nulls before:", hrdata['employee_type'].isnull().sum())

# Address missing values
hrdata['employee_type'] = hrdata['employee_type'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['employee_type'].isnull().sum())

Nulls before: 7
Nulls after: 0


office_location

In [1310]:
#Remove leading and trailing  spaces and  change to title cae
hrdata['office_location']=hrdata['office_location'].str.title().str.strip()

#Identify unique categories in the column
print(hrdata['office_location'].unique())

#Count unique categories in the column
hrdata['office_location'].value_counts()

<ArrowStringArray>
[       'Nairob',       'Nairobi',        'Berlin',         'Tokyo',
 'San Francisco',        'London',      'New York', 'San Fransisco',
         'Londn',             nan,         'Tokio',            'Sf',
        'Remote',         'Berln']
Length: 14, dtype: str


office_location
San Francisco    143
Nairobi          136
Tokyo            134
London           133
New York         130
Berlin           128
San Fransisco     11
Londn             11
Tokio              7
Nairob             5
Sf                 4
Remote             4
Berln              3
Name: count, dtype: int64

In [1311]:
#Standardize inconsistent Categories
hrdata['office_location']=hrdata['office_location'].replace({
    "Nairob":"Nairobi","NAIROBI":"Nairobi",
    "San Fransisco":"San Francisco","SF":"San Francisco",
    "Londn":"London",
    "Tokio":"Tokyo",
    "Berln":"Berlin",
    " nan":"Unknown"
})

#Check data type
print(type('office_location'))

<class 'str'>


In [1312]:
# Check current null count
print("Nulls before:", hrdata['office_location'].isnull().sum())

# Address missing values
hrdata['office_location'] = hrdata['office_location'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['office_location'].isnull().sum())

Nulls before: 13
Nulls after: 0


remote_work_status

In [1313]:
#Remove leading and trailing  spaces and  chnage to title cae
hrdata['remote_work_status']=hrdata['remote_work_status'].str.title().str.strip()

#Identify unique categories in the column
print(hrdata['remote_work_status'].unique())

#Count unique categories in the column
hrdata['remote_work_status'].value_counts()

<ArrowStringArray>
[     'On-Site',       'Hybrid', 'Fully Remote',            nan,
       'Remote',      'On Site',       'Onsite',       'Hybird']
Length: 8, dtype: str


remote_work_status
On-Site         286
Hybrid          285
Fully Remote    254
Onsite            9
Remote            8
On Site           5
Hybird            4
Name: count, dtype: int64

In [1314]:
#Standardize inconsistent Categories
hrdata['remote_work_status']=hrdata['remote_work_status'].replace({
    "On site":"On-Site","on-site":"On-Site","Onsite":"On-Site",
    "Fully Remote":"Remote","fully remote":"Remote",
    "Hybird":"Hybrid","hybrid":"Hybrid",
    "nan":"Unknown"
})

#Check data type
print(type('remote_work_status'))

<class 'str'>


In [1315]:
# Check current null count
print("Nulls before:", hrdata['remote_work_status'].isnull().sum())

# Address missing values
hrdata['remote_work_status'] = hrdata['remote_work_status'].fillna('Unknown')

# Verify null count is now 0
print("Nulls after:", hrdata['remote_work_status'].isnull().sum())

Nulls before: 11
Nulls after: 0


**Step 4. Numerical Columns Cleaning and Standardization**

- Fix  word numbers,nan,N/A,'', & Data types

**Identify All Numeric Columns in The Dataset**

- Columns like age, bonus, project_count, and manager_feedback_score are numbers.
-  They are still trapped as text (object or str types)  because they contain  dirty characters (like words, spaces, typos, or NaN labels).

In [1316]:
# Select and display the names of all true numerical columns
numeric_columns = hrdata.select_dtypes(include=['number']).columns
list(numeric_columns)


['annual_training_hours']

In [1317]:
# List of all columns and their data types
hrdata.dtypes


employee_id                object
first_name                    str
last_name                     str
department                    str
salary                     object
hire_date                  object
age                        object
gender                        str
performance_score          object
full_time                     str
bonus                      object
marital_status                str
education_level               str
work_experience_years      object
employee_type                 str
office_location               str
project_count              object
last_promotion_year        object
remote_work_status            str
annual_training_hours     float64
manager_feedback_score     object
dtype: object

**Column By Column Clean Up**

Salary

In [1318]:
# Count how many rows have missing or blank salary values
missing_salaries = hrdata['salary'].isnull().sum()

print(f"Number of missing values in Salary: {missing_salaries}")

# List out every unique variation present in the salary column
unique_salaries = hrdata['salary'].unique()

print(unique_salaries[:10]) #Output the first 10 rows

Number of missing values in Salary: 9
[74291 93350 112020 34646 69713 '97936' 74331 52185 114263 '82107']


In [1319]:
# Convert salary values to strings, then use regex to remove:
# KES, $, commas, and whitespace, leaving only the numeric value.
#
# Regex: r'KES|\$|,|\s'
# - KES = removes "KES"
# - \$  = removes the "$" symbol
# - ,   = removes commas
# - \s  = removes spaces/whitespace
# - |   = means "OR"
# - regex=True = tells Pandas to treat the pattern as a regex

hrdata['salary'] = hrdata['salary'].astype(str).str.replace(r'KES|\$|,|\s', '', regex=True)


# Convert the clean text back into floats (any remaining bad text or raw 'nan' blanks safely become NaN)
hrdata['salary'] = pd.to_numeric(hrdata['salary'], errors='coerce')

# Print your brand new, perfectly standardized unique number values to confirm
print("Missing values now:", hrdata['salary'].isnull().sum())
print("\nFirst 10 clean numbers:\n", hrdata['salary'].dropna().unique()[:10])


Missing values now: 9

First 10 clean numbers:
 [ 74291.  93350. 112020.  34646.  69713.  97936.  74331.  52185. 114263.
  82107.]


In [1320]:
# Calculate the median salary (ignoring the missing ones for now)
median_salary = hrdata['salary'].median()
print(f"Median salary is: {median_salary}\n")

# Fill the 9 missing spaces with that median value
hrdata['salary'] = hrdata['salary'].fillna(median_salary)

# Double-check that missing values are completely gone
print("Missing values remaining in Salary:", hrdata['salary'].isnull().sum())

# Force the text-cleaned strings into actual decimal numbers (floats)
hrdata['salary'] = pd.to_numeric(hrdata['salary'], errors='coerce')



Median salary is: 76008.0

Missing values remaining in Salary: 0


hire_date

- To fix this, we use pd.to_datetime().

In [1321]:
# Force the column to a date format
# 'errors="coerce"' automatically converts text like "not available" into blank NaT (Not a Time) flags
hrdata['hire_date'] = pd.to_datetime(hrdata['hire_date'], errors='coerce')

# Check for missing or broken dates after conversion
missing_dates = hrdata['hire_date'].isnull().sum()
print(f"Number of broken or missing dates: {missing_dates}")

# Retain invalid/missing dates as NaT instead of replacing  missing values
hrdata['hire_date'] = pd.to_datetime(hrdata['hire_date'], errors='coerce')


# View a quick sample to see how clean they look now
print("\nSample of clean dates:")
print(hrdata['hire_date'].dropna().head())


Number of broken or missing dates: 44

Sample of clean dates:
0   2002-09-14
1   2008-11-12
2   2017-07-13
3   2014-07-27
4   2010-07-12
Name: hire_date, dtype: datetime64[us]


age  

In [1322]:
# Convert to string, lowercase it, and change written words to numbers(thirty)
hrdata['age'] = hrdata['age'].astype(str).str.lower().str.strip()
hrdata['age'] = hrdata['age'].str.replace('thirty', '30')

# Extract only numbers (this handles decimals like 35.5 and strips any accidental text)
hrdata['age'] = hrdata['age'].str.extract(r'(\d+\.?\d*)')

#Element Breakdown.
# str.extract(...): Tells Python to find a specific pattern, pull it out, and discard everything else.
# r: Stands for "Raw String." It ensures the symbols inside the pattern are read as text commands, not regular code.
# \d+: Finds the main numbers (e.g., 35 in 35.5).\.
# ?: Safely allows for an optional decimal point.
# \d*: Finds any numbers after the decimal point (e.g., .5 in 35.5).

# 3. Force to a numerical float type (errors='coerce' turns empty cells/bad text into NaN)
hrdata['age'] = pd.to_numeric(hrdata['age'], errors='coerce')

# 4. Fix extreme outliers (e.g., age 4 or age 99) by turning them into NaN temporarily
# We assume a normal working age is between 18 and 75
hrdata.loc[(hrdata['age'] < 18) | (hrdata['age'] > 75), 'age'] = np.nan

# 5. Fill all missing values and corrected outliers with the company median age
company_median_age = hrdata['age'].median()
hrdata['age'] = hrdata['age'].fillna(company_median_age).astype(int) # Convert to integer at the end

# 6. Verify our work
print(f"Company Median Age used for blanks: {int(company_median_age)}")
print("Missing values remaining in Age:", hrdata['age'].isnull().sum())
print("\nSummary of your clean Age column:")
print(hrdata['age'].describe())


Company Median Age used for blanks: 41
Missing values remaining in Age: 0

Summary of your clean Age column:
count    862.000000
mean      41.903712
std       12.175784
min       22.000000
25%       31.000000
50%       41.000000
75%       52.000000
max       64.000000
Name: age, dtype: float64


performance_score

In [1323]:
# Force directly to numbers (this automatically converts "Excellent", "Poor", "N/A" to NaN)
hrdata['performance_score'] = pd.to_numeric(hrdata['performance_score'], errors='coerce')

# Fill the new blank cells with the median score of the company
median_score = hrdata['performance_score'].median()
hrdata['performance_score'] = hrdata['performance_score'].fillna(median_score).astype(int)

print("Missing values remaining:", hrdata['performance_score'].isnull().sum())


Missing values remaining: 0


bonus

In [1324]:
# Strip out currency tags (KES, $) commas, and spaces using a regular expression
hrdata['bonus'] = hrdata['bonus'].astype(str).str.replace(r'[\$,KRES\s-]', '', regex=True)

# Force the column to be a number 
# errors='coerce' turns words like "N/A" into a true missing value (NaN)

hrdata['bonus'] = pd.to_numeric(hrdata['bonus'], errors='coerce')

# Fill all missing values/NaNs with 0 (Assuming no data means $0 bonus)
hrdata['bonus'] = hrdata['bonus'].fillna(0)

# Round the column to 2 decimal places since it represents money
hrdata['bonus'] = hrdata['bonus'].round(2)

# 5. Verify your work
print("Missing values remaining in Bonus:", hrdata['bonus'].isnull().sum())
print("\nA sample of your clean bonus values:")
print(hrdata['bonus'].head(10))


Missing values remaining in Bonus: 0

A sample of your clean bonus values:
0    5324.53
1    5037.68
2    1087.86
3    4817.13
4    3231.82
5    3108.36
6    2612.63
7    3659.02
8    3854.76
9    3433.37
Name: bonus, dtype: float64


work_experience_years

In [1325]:
# Convert to string, lowercase it, and remove extra spaces
hrdata['work_experience_years'] = hrdata['work_experience_years'].astype(str).str.lower().str.strip()

# Extract only numbers and decimals (strips away words like "years")
hrdata['work_experience_years'] = hrdata['work_experience_years'].str.extract(r'(\d+\.?\d*)')

# Force to a numerical float type
hrdata['work_experience_years'] = pd.to_numeric(hrdata['work_experience_years'], errors='coerce')

# Handle negative numbers: Treat any value below 0 as a missing value (NaN)
# 1. hrdata.loc[rows, column]: Selects specific matching rows within a targeted column.
# 2. hrdata['work_experience_years'] < 0: Finds any rows where the experience number is negative.
# 3. = np.nan: Overwrites those negative values with a true missing value marker (NaN).
hrdata.loc[hrdata['work_experience_years'] < 0, 'work_experience_years'] = np.nan

# Fill missing values (and the converted negative numbers) with the median experience level
median_experience = hrdata['work_experience_years'].median()
hrdata['work_experience_years'] = hrdata['work_experience_years'].fillna(median_experience).astype(int)

# Verify our work
print(f"Company Median Experience used for blanks: {int(median_experience)} years")
print("Missing values remaining:", hrdata['work_experience_years'].isnull().sum())
print("\nA sample of your clean work experience values:")
print(hrdata['work_experience_years'].head(15))


Company Median Experience used for blanks: 20 years
Missing values remaining: 0

A sample of your clean work experience values:
0     13
1     39
2     20
3     32
4     10
5     26
6     19
7     29
8     13
9     34
10    20
11    18
12    34
14    16
15    23
Name: work_experience_years, dtype: int64


project_count

In [1326]:
# .replace({'ten': '10'}): Swaps the written word "ten" with the string digit "10".
hrdata['project_count'] = hrdata['project_count'].astype(str).str.lower().str.strip().replace({'ten': '10'})

# pd.to_numeric(..., errors='coerce'): Converts text numbers to math numbers; breaks/typos become NaN.
hrdata['project_count'] = pd.to_numeric(hrdata['project_count'], errors='coerce')

# .fillna(...): Replaces any blank cells or created NaNs with the company median project count.
hrdata['project_count'] = hrdata['project_count'].fillna(hrdata['project_count'].median()).astype(int)


In [1327]:
hrdata.dtypes

employee_id                       object
first_name                           str
last_name                            str
department                           str
salary                           float64
hire_date                 datetime64[us]
age                                int64
gender                               str
performance_score                  int64
full_time                            str
bonus                            float64
marital_status                       str
education_level                      str
work_experience_years              int64
employee_type                        str
office_location                      str
project_count                      int64
last_promotion_year               object
remote_work_status                   str
annual_training_hours            float64
manager_feedback_score            object
dtype: object

last_promotion_year 

In [1328]:
# 1. pd.to_numeric(..., errors='coerce'): Converts clean years to numbers. 
#    Automatically turns words like "Never" or "N/A" into a true missing blank (NaN).
hrdata['last_promotion_year'] = pd.to_numeric(hrdata['last_promotion_year'], errors='coerce')

# 2. .fillna(0): Replaces the new NaN blanks with 0.
#    In HR analytics, a promotion year of 0 logically indicates the employee has never been promoted.
hrdata['last_promotion_year'] = hrdata['last_promotion_year'].fillna(0).astype(int)


annual_training_hours 

In [1329]:
# .median(): Calculates the middle value of the training hours to use as a fair fallback.
training_median = hrdata['annual_training_hours'].median()

# .fillna(...): Replaces any blank cells or missing records with that median value.
hrdata['annual_training_hours'] = hrdata['annual_training_hours'].fillna(training_median)

# .astype(int): Converts the column from decimal format (float) into clean, whole numbers (integers).
hrdata['annual_training_hours'] = hrdata['annual_training_hours'].astype(int)


manager_feedback_score 

In [1330]:
# Created a dictionary to convert feedback words into standard numbers on a 1-5 scale.
score_mapping = {'Excellent': 5, 'Good': 4, 'Poor': 1, 'N/A': np.nan, 'None': np.nan}

# .replace(...): Swaps the words ("Good", "Excellent") with their numeric equivalents.
hrdata['manager_feedback_score'] = hrdata['manager_feedback_score'].astype(str).str.strip().replace(score_mapping)

# pd.to_numeric(..., errors='coerce'): Forces the text numbers into math-ready float format.
hrdata['manager_feedback_score'] = pd.to_numeric(hrdata['manager_feedback_score'], errors='coerce')

# .fillna(...): Fills empty rows with the column median so your final metrics remain unskewed.
feedback_median = hrdata['manager_feedback_score'].median()
hrdata['manager_feedback_score'] = hrdata['manager_feedback_score'].fillna(feedback_median).round(1)


**Step 5. Final Quality Report**

In [1331]:
quality_report = pd.DataFrame({
    "Column": hrdata.columns,
    "Data_Type": hrdata.dtypes.astype(str),
    "Total_Rows": len(hrdata),
    "Missing_Values": hrdata.isna().sum().values,
    "Missing_Percentage": (hrdata.isna().mean() * 100).round(2).values,
    "Unique_Values": hrdata.nunique().values,
})

quality_report


,Column,Data_Type,Total_Rows,Missing_Values,Missing_Percentage,Unique_Values
employee_id,employee_id,object,862,0,0.00,852
first_name,first_name,str,862,11,1.28,91
last_name,last_name,str,862,9,1.04,60
department,department,str,862,0,0.00,8
salary,salary,float64,862,0,0.00,846
hire_date,hire_date,datetime64[us],862,44,5.10,760
age,age,int64,862,0,0.00,43
gender,gender,str,862,0,0.00,3
performance_score,performance_score,int64,862,0,0.00,11
full_time,full_time,str,862,0,0.00,3


**Step 6.Data Export**

In [ ]:
# Save cleaned DataFrame to a new Excel file
hrdata.to_excel('HR_Cleaned_Data.xlsx', index=False)